In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
from helper import *

from sklearn.model_selection import GridSearchCV

c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import mlflow
import mlflow.pytorch

In [3]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [4]:
mlflow.set_experiment("Clasificador_Imagenes")

2026/05/30 20:54:34 INFO mlflow.tracking.fluent: Experiment with name 'Clasificador_Imagenes' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:///c:/Users/Camila/OneDrive/Escritorio/Redes '
 'Neuronales/Skin-dataset-classification-CS2026/mlruns/2'), creation_time=1780185274539, experiment_id='2', last_update_time=1780185274539, lifecycle_stage='active', name='Clasificador_Imagenes', tags={}, trace_location=None, workspace='default'>

In [5]:
def log_classification_report(model, loader, writer, device, classes, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    # Guardar localmente y subir a MLflow
    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)

    cls_report = classification_report(all_labels, all_preds, target_names=classes)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    # También loguear texto del reporte
    with open(f"classification_report_{prefix}_epoch_{step}.txt", "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(f.name)
    os.remove(f.name)


In [6]:
# Entrenamiento y validación
def evaluate(model, loader, writer, device, classes, epoch=None, prefix="val"):
    log_classification_report(model, loader, writer, device, classes, step=epoch , prefix="val")
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Loguear imágenes del primer batch
            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

In [7]:
# Paths
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val/"

In [8]:
# Crear directorio de logs de tensorboard
log_dir = "runs/experimento_skin"
writer = SummaryWriter(log_dir=log_dir)

In [9]:
np.random.rand()

0.9469405666175927

In [ ]:
hparams_space= {
    "model": ("MLPClassifier"),
    "input_size":  [32,64,128],
    "batch_size": [16,64,128],
    "lr": [1e-2,1e-3,1e-4],
    "epochs": 200,
    "optimizer":  ["Adam", "SGD"],
    "HFlip": [0.0,0.5],
    "VFlip": [0.0,0.5],
    "RBContrast": [0.0, 0.5],
    "loss_fn": "CrossEntropyLoss",
    "train_dir": train_dir,
    "val_dir": val_dir,
    "es_patience": 5,
    "dropout": [0.0,0.1,0.2,0.3],
}

In [13]:
modelnbr = 0
for input_size in hparams_space["input_size"]:
    for batch_size in hparams_space["batch_size"]:
        for lr in hparams_space["lr"]:
            for optimizer in hparams_space["optimizer"]:
                for HFlip in hparams_space["HFlip"]:
                    for VFlip in hparams_space["VFlip"]:
                        for RBContrast in hparams_space["RBContrast"]:
                            for dropout in hparams_space["dropout"]:
                                if np.random.rand() < 0.1:
                                    print(f"modelo número: {modelnbr}", end = "\r")
                                    modelnbr += 1
                                    hparams= {
                                        "model": ("MLPClassifier"),
                                        "input_size":  input_size,
                                        "batch_size": batch_size,
                                        "lr": lr,
                                        "epochs": 200,
                                        "optimizer": optimizer,
                                        "HFlip": HFlip,
                                        "VFlip": VFlip,
                                        "RBContrast": RBContrast,
                                        "loss_fn": "CrossEntropyLoss",
                                        "train_dir": train_dir,
                                        "val_dir": val_dir,
                                        "es_patience": 5,
                                        "dropout": dropout,
                                    }
                                    train_transform = A.Compose([
                                        A.Resize(hparams["input_size"], hparams["input_size"]),
                                        A.HorizontalFlip(p=hparams["HFlip"]),
                                        A.VerticalFlip(p=hparams["VFlip"]),
                                        A.RandomBrightnessContrast(p=hparams["RBContrast"]),
                                        A.Normalize(),
                                        ToTensorV2()
                                    ])
                                    val_test_transform = A.Compose([
                                        A.Resize(hparams["input_size"], hparams["input_size"]),
                                        A.Normalize(),
                                        ToTensorV2()
                                    ])
                                    train_dataset = CustomImageDataset(train_dir, transform=train_transform)
                                    val_dataset   = CustomImageDataset(val_dir, transform=val_test_transform)
                                    batch_size = hparams["batch_size"]
                                    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                                    val_loader   = DataLoader(val_dataset, batch_size=batch_size)
                                    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                                    num_classes = len(set(train_dataset.labels))
                                    model = MLPClassifier(num_classes=num_classes, input_size = hparams["input_size"]**2*3, dropout = hparams["dropout"]).to(device)
                                    criterion = nn.CrossEntropyLoss()
                                    optimizer = optim.Adam(model.parameters(), lr=hparams["lr"]) if hparams["optimizer"]=="Adam" else optim.SGD(model.parameters(), lr=hparams["lr"])
                                    hparams["count_params"] = count_parameters(model)
                                    with mlflow.start_run():
                                        # Log hiperparámetros
                                        mlflow.log_params(hparams)
                                        best_val_acc = 0
                                        best_val_loss = 0
                                        best_train_acc = 0
                                        best_train_loss = 0
                                        best_epoch = 0
                                        for epoch in range(hparams["epochs"]):
                                            model.train()
                                            running_loss = 0.0
                                            correct, total = 0, 0
                                        
                                            for images, labels in train_loader:
                                                images, labels = images.to(device), labels.to(device)
                                        
                                                optimizer.zero_grad()
                                                outputs = model(images)
                                                loss = criterion(outputs, labels)
                                                loss.backward()
                                                optimizer.step()
                                        
                                                running_loss += loss.item()
                                                _, preds = torch.max(outputs, 1)
                                                correct += (preds == labels).sum().item()
                                                total += labels.size(0)
                                        
                                            train_loss = running_loss / len(train_loader)
                                            train_acc = 100.0 * correct / total
                                            val_loss, val_acc = evaluate(model, val_loader, writer, device,train_dataset.label_encoder.classes_,epoch=epoch, prefix="val")
                                        
                                            #print(f"Epoch {epoch+1}:")
                                            #print(f"  Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
                                            #print(f"  Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")
                                        
                                            writer.add_scalar("train/loss", train_loss, epoch)
                                            writer.add_scalar("train/accuracy", train_acc, epoch)
                                        
                                            # Log en MLflow
                                            mlflow.log_metrics({
                                                "train_loss": train_loss,
                                                "train_accuracy": train_acc,
                                                "val_loss": val_loss,
                                                "val_accuracy": val_acc
                                            }, step=epoch)
                                            if val_acc > best_val_acc:
                                                best_val_acc = val_acc
                                                best_val_loss = val_loss
                                                best_train_acc = train_acc
                                                best_train_loss = train_loss
                                                best_epoch = epoch
                                                # Guardar modelo
                                                torch.save(model.state_dict(), "mlp_model.pth")
                                                #print("Modelo guardado como 'mlp_model.pth'")
                                                mlflow.log_artifact("mlp_model.pth")
                                                mlflow.pytorch.log_model(model, artifact_path="pytorch_model")
                                            elif epoch > best_epoch + hparams["es_patience"]:
                                                #print("Early Stopping")
                                                break
                                                
                                        mlflow.log_metrics({
                                                "train_loss": best_train_loss,
                                                "train_accuracy": best_train_acc,
                                                "val_loss": best_val_loss,
                                                "val_accuracy": best_val_acc,
                                                "best_epoch": best_epoch
                                            }, step=epoch+1)                                                
                                        

2026/05/31 03:00:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/31 03:00:31 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision 

c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

...

c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

2026/05/31 11:33:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/31 11:33:40 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/31 11:34:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/31 11:34:05 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
c:\Users

c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Camila\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

OSError: [WinError 112] Espacio en disco insuficiente

In [14]:
runs = mlflow.search_runs(order_by=["metrics.val_accuracy DESC"])
top_10 = runs.head(int(len(runs) * 0.10))
# Analizar qué HP tienen en común:
top_10[["params.lr", "params.optimizer", "params.dropout"]].describe()

,params.lr,params.optimizer,params.dropout
count,10,10,10
unique,3,3,4
top,0.001,Adam,0.0
freq,4,7,3


In [15]:
print(runs[["params.lr", "params.optimizer", "params.dropout", 
            "metrics.val_accuracy", "metrics.best_epoch"]]
      .sort_values("metrics.val_accuracy", ascending=False)
      .head(10))

  params.lr                                   params.optimizer params.dropout  \
0    0.0001                                               Adam            0.2   
1    0.0001                                               Adam            0.0   
2     0.001                                               Adam            0.0   
3     0.001                                               Adam            0.1   
4    0.0001                                               Adam            0.3   
5      0.01                                                SGD            0.1   
6     0.001                                               Adam            0.0   
7      0.01  SGD (\nParameter Group 0\n    dampening: 0\n  ...            0.3   
8      0.01  SGD (\nParameter Group 0\n    dampening: 0\n  ...            0.3   
9     0.001                                               Adam            0.2   

   metrics.val_accuracy  metrics.best_epoch  
0             68.508287                28.0  
1             67

In [23]:
import mlflow

runs = mlflow.search_runs(order_by=["metrics.val_accuracy DESC"])
mejor = runs.iloc[0]
print(mejor)

run_id                                      030d1d3d0ca44c9e871e78c96a419e0c
experiment_id                                                              2
status                                                              FINISHED
artifact_uri               file:///c:/Users/Camila/OneDrive/Escritorio/Re...
start_time                                  2026-05-31 00:39:10.362000+00:00
end_time                                    2026-05-31 00:47:24.826000+00:00
metrics.val_accuracy                                               68.508287
metrics.best_epoch                                                      28.0
metrics.val_loss                                                    0.971327
metrics.train_accuracy                                             82.352941
metrics.train_loss                                                  0.504371
params.optimizer                                                        Adam
params.VFlip                                                             0.0

In [25]:
import json
from datetime import datetime

resumen = {
    "fecha": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "run_id": mejor["run_id"],
    "resultado": {
        "val_accuracy": round(mejor["metrics.val_accuracy"], 4),
        "val_loss": round(mejor["metrics.val_loss"], 4),
        "train_accuracy": round(mejor["metrics.train_accuracy"], 4),
        "train_loss": round(mejor["metrics.train_loss"], 4),
        "best_epoch": int(mejor["metrics.best_epoch"]),
    },
    "hiperparametros": {
        "model": mejor["params.model"],
        "input_size": int(mejor["params.input_size"]),
        "batch_size": int(mejor["params.batch_size"]),
        "lr": float(mejor["params.lr"]),
        "optimizer": mejor["params.optimizer"],
        "dropout": float(mejor["params.dropout"]),
        "HFlip": float(mejor["params.HFlip"]),
        "VFlip": float(mejor["params.VFlip"]),
        "RBContrast": float(mejor["params.RBContrast"]),
        "es_patience": int(mejor["params.es_patience"]),
        "epochs": int(mejor["params.epochs"]),
        "count_params": int(mejor["params.count_params"]),
    }
}

with open("mejor_modelo.json", "w") as f:
    json.dump(resumen, f, indent=4)

print(json.dumps(resumen, indent=4))

{
    "fecha": "2026-05-31 12:41",
    "run_id": "030d1d3d0ca44c9e871e78c96a419e0c",
    "resultado": {
        "val_accuracy": 68.5083,
        "val_loss": 0.9713,
        "train_accuracy": 82.3529,
        "train_loss": 0.5044,
        "best_epoch": 28
    },
    "hiperparametros": {
        "model": "MLPClassifier",
        "input_size": 32,
        "batch_size": 16,
        "lr": 0.0001,
        "optimizer": "Adam",
        "dropout": 0.2,
        "HFlip": 0.0,
        "VFlip": 0.0,
        "RBContrast": 0.0,
        "es_patience": 5,
        "epochs": 200,
        "count_params": 1640201
    }
}


In [ ]:
import os
import hashlib
from pathlib import Path

def get_hash(filepath):
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

train_dir = Path("data/Split_smol/train")
val_dir = Path("data/Split_smol/val")

# Hashear todas las imágenes
train_hashes = {get_hash(f): f for f in train_dir.rglob("*") if f.is_file()}
val_hashes   = {get_hash(f): f for f in val_dir.rglob("*") if f.is_file()}

# Buscar duplicados
duplicados = set(train_hashes.keys()) & set(val_hashes.keys())

print(f"Imágenes en train: {len(train_hashes)}")
print(f"Imágenes en val:   {len(val_hashes)}")
print(f"Duplicados:        {len(duplicados)}")


Imágenes en train: 696
Imágenes en val:   181
Duplicados:        33


In [21]:
import shutil

def redistribuir_clase(clase, n_mover):
    train_clase = train_dir / clase
    val_clase = val_dir / clase
    
    # Agarrar imágenes de train que NO son duplicadas
    no_duplicadas = [f for f in train_clase.glob("*") 
                     if f.is_file() and get_hash(f) not in duplicados]
    
    # Mover las primeras n_mover
    for f in no_duplicadas[:n_mover]:
        shutil.move(str(f), str(val_clase / f.name))
        print(f"  Movido: {f.name}")

# Atopic Dermatitis: val queda vacío, mover 20 de train
print("Atopic Dermatitis:")
redistribuir_clase('Atopic Dermatitis', 20)

# Tinea: val queda con 8 después de borrar duplicados, mover 12 más
print("\nTinea Ringworm Candidiasis:")
redistribuir_clase('Tinea Ringworm Candidiasis', 12)

Atopic Dermatitis:
  Movido: 1_21.jpg
  Movido: 1_22.jpg
  Movido: 1_31.jpg
  Movido: 1_32.jpg
  Movido: 1_45.jpg
  Movido: 1_46.jpg
  Movido: 1_47.jpg
  Movido: 1_48.jpg
  Movido: 1_49.jpg
  Movido: 1_51.jpg
  Movido: 1_52.jpg
  Movido: 1_53.jpg
  Movido: 1_55.jpg
  Movido: 1_57.jpg
  Movido: 1_58.jpg
  Movido: 1_59.jpg
  Movido: 1_60.jpg
  Movido: 1_62.jpg
  Movido: 1_63.jpg
  Movido: 1_64.jpg

Tinea Ringworm Candidiasis:
  Movido: 148018-1.jpg
  Movido: 148018.jpg
  Movido: 15997525228_30a0f1256f_z.jpg
  Movido: 300px-Ringworm_on_the_arm,_or_tinea_corporis_due_to_Trichophyton_mentagrophytes_PHIL_2938_lores.jpg
  Movido: afp20030101p101-f1.jpg
  Movido: afp20080515p1415-f2.jpg
  Movido: aug_0_11046_2008_9101_Fig2_HTML.jpg
  Movido: aug_0_1200px-Oberflachlicher_Hautpilz.jpg
  Movido: aug_0_15436_lores.jpg
  Movido: aug_0_241-1.jpg
  Movido: aug_0_300px-Ringworm_on_the_arm,_or_tinea_corporis_due_to_Trichophyton_mentagrophytes_PHIL_2938_lores.jpg
  Movido: aug_0_afp20030101p101-f1.jpg
